<a href="https://colab.research.google.com/github/goyaltanisha447-pixel/amazon_masterclass_project/blob/main/amazon_masterclass_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile requirments.txt
!pip install pandas numpy matplotlib seaborn scikit-learn
!pip install sentence-transformers faiss-cpu
!pip install transformers torch wordcloud
!pip install --upgrade transformers


Overwriting requirments.txt


In [ ]:
%%writefile python.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
import faiss

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from transformers import pipeline

from wordcloud import WordCloud
import re
from google.colab import files
uploaded = files.upload()
import pandas as pd

df = pd.read_csv("Dataset-SA.csv")
df.head()
df.info()
df.isnull().sum()
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["clean_review"] = df["Review"].apply(clean_text)
plt.figure(figsize=(6,4))
sns.countplot(x="Rate", data=df)
plt.title("Rating Distribution")
plt.show()
df["Rate"].value_counts()
text = " ".join(df["Review"].astype(str))

from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400).generate(text)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud)
plt.axis("off")
plt.show()
df["review_length"] = df["Review"].astype(str).apply(len)

plt.hist(df["review_length"], bins=50)
plt.title("Review Length Distribution")
plt.show()
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df["clean_review"].tolist(),
    show_progress_bar=True
)
dimension = embeddings.shape[1]
embeddings_normalized = embeddings.copy()

faiss.normalize_L2(embeddings_normalized)

index = faiss.IndexFlatIP(dimension)

index.add(embeddings_normalized)
index.ntotal
def semantic_search(query, top_k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = df.iloc[indices[0]]

    return results[["product_name", "Review", "Rate"]]
    semantic_search("energy efficient air conditioner")
    pca = PCA(n_components=2)

reduced_embeddings = pca.fit_transform(embeddings[:2000])

plt.scatter(reduced_embeddings[:,0], reduced_embeddings[:,1], alpha=0.5)
plt.title("PCA Visualization of Embeddings")
plt.show()
def evaluate(query, k):

    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, k)

    retrieved = df.iloc[indices[0]]

    ground_truth = df[df["clean_review"].str.contains(query.lower())]

    relevant = set(ground_truth.index)
    retrieved_ids = set(retrieved.index)

    true_positive = len(retrieved_ids.intersection(relevant))

    precision = true_positive / k
    recall = true_positive / len(relevant) if len(relevant) > 0 else 0

    return precision, recall
    query = "battery life"

for k in [1,3,5,10]:
    p,r = evaluate(query,k)
    print(f"K={k} | Precision={p:.3f} | Recall={r:.3f}")
    from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)
def rag_pipeline(query):

    docs = semantic_search(query, top_k=3)

    context = " ".join(docs["Review"].astype(str).tolist())

    prompt = f"""
    Reviews:
    {context}

    Question: {query}
    Answer:
    """

    response = generator(prompt, max_length=100)

    return response[0]["generated_text"]

rag_pipeline("Is the battery life good?")
semantic_search("good camera phone")

semantic_search("fast charging battery")

semantic_search("comfortable headphones")


Writing python.py
